# Lab 4 - Compound AI Agents for Aircraft Analytics

Lab 4 puts two query engines behind one natural language interface. A **Genie space** answers questions about sensor telemetry with SQL over the Lakehouse. **Neo4j MCP** answers questions about relationships with Cypher over the graph. A **Supervisor Agent** reads each question and picks the right one, and for questions that need both it queries each in turn and combines the answers.

![Lab Architecture Overview](https://raw.githubusercontent.com/neo4j-partners/databricks-neo4j-workshop/main/images/lab-architecture-overview.png)

## Part A is yours to build. Part B is an instructor demo.

| | What you do | Time |
|---|---|---|
| **Part A: Genie space** | You build it in the Databricks UI, following this notebook | ~30 min |
| **Part B: Supervisor Agent** | You watch. The instructor builds it on screen | ~20 min |

**Part A** creates the Genie space over the shared Lakehouse tables in Unity Catalog. Lab 5 uses that Genie space as one of its three tools, so Part A is on your path.

**Part B** runs against the instructor's demo workspace and the instructor's Aura instance. You need no Aura instance, no MCP connection, and no OAuth credential to watch it.

## Prerequisites

For Part A:
- A Databricks workspace with AI/BI Genie access
- A Serverless SQL Warehouse you can select
- The workshop catalog `databricks-neo4j-workshop` loaded by your admin

Part B needs nothing from you.

**Recommended:** finish **Lab 2** first. Part A queries shared Lakehouse tables rather than your own Aura instance, but Lab 2 teaches you the data model: aircraft topology, sensor relationships, flights, and maintenance events. Labs 5 and 6 do need your own loaded instance.

---

# Part A: Genie space for Aircraft Sensor Analytics

You build an AI/BI Genie space that answers natural language questions over aircraft sensor telemetry. This Genie space becomes one of the sub-agents in the multi-agent system, and one of the three tools in Lab 5.

## Step 1: Explore the Lakehouse Data

Your workshop admin pre-loaded a set of tables into Unity Catalog holding the Aircraft Digital Twin sensor telemetry. This is the data you point Genie at, so look at it first.

You can browse it in the UI:

1. Click **Catalog** in the left sidebar.
2. Expand **databricks-neo4j-workshop > aircraft**.
3. Click a table such as `sensor_readings` and open the **Sample Data** tab.

![Lakehouse sensor_readings table in Unity Catalog](https://raw.githubusercontent.com/neo4j-partners/databricks-neo4j-workshop/main/Lab_4_Compound_AI_Agents/images/lakehouse_sensor_readings.png)

The four tables Genie needs:

| Table | Description |
|-------|-------------|
| `aircraft` | Fleet inventory: tail numbers, models, manufacturers, operators |
| `systems` | Aircraft systems: engines, avionics, hydraulics |
| `sensors` | Sensor metadata: EGT, vibration, N1 speed, fuel flow |
| `sensor_readings` | Telemetry every 4 hours over 90 days, July to September 2024 |

They form a join chain, `sensor_readings` to `sensors` to `systems` to `aircraft`, that connects a raw telemetry value all the way up to fleet metadata. Genie walks that chain to answer questions.

The cells below query the same tables from this notebook. Run them.

In [ ]:
# The workshop catalog and schema. Both names come from lab/workshop.py,
# the one definition of every object this course creates.
CATALOG = "databricks-neo4j-workshop"
SCHEMA = "aircraft"

# The catalog name contains hyphens, so it has to be backticked in SQL.
FQN = f"`{CATALOG}`.`{SCHEMA}`"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

print(f"Reading from {FQN}")
display(spark.sql(f"SHOW TABLES IN {FQN}"))

### The telemetry itself

`sensor_readings` is the fact table: one row per sensor per timestamp, 155,520 rows in total.

In [ ]:
display(spark.sql(f"SELECT * FROM {FQN}.sensor_readings LIMIT 20"))

In [ ]:
# How much data, over what window, across how many sensors.
display(
    spark.sql(f"""
        SELECT
            COUNT(*)                  AS reading_count,
            COUNT(DISTINCT sensor_id) AS sensor_count,
            MIN(timestamp)            AS first_reading,
            MAX(timestamp)            AS last_reading
        FROM {FQN}.sensor_readings
    """)
)

### The dimensions around it

`sensors` says what each reading measures and in what unit. `systems` and `aircraft` say what it is bolted to.

In [ ]:
display(
    spark.sql(f"""
        SELECT type, unit, COUNT(*) AS sensor_count
        FROM {FQN}.sensors
        GROUP BY type, unit
        ORDER BY type
    """)
)

In [ ]:
display(
    spark.sql(f"""
        SELECT manufacturer, model, operator, COUNT(*) AS aircraft_count
        FROM {FQN}.aircraft
        GROUP BY manufacturer, model, operator
        ORDER BY manufacturer, model
    """)
)

### The join chain, walked once by hand

This is the query shape Genie generates for you. Average EGT per model, reading telemetry values and grouping them by fleet metadata three joins away.

Note the spread across models. A fleet-wide EGT average mixes engines with different normal ranges, so it means little on its own. That is why the Genie instructions in Step 3 tell it to group by model before comparing EGT.

In [ ]:
display(
    spark.sql(f"""
        SELECT
            a.model,
            ROUND(AVG(r.value), 1) AS avg_egt,
            COUNT(*)               AS reading_count
        FROM {FQN}.sensor_readings r
        JOIN {FQN}.sensors  sen ON r.sensor_id = sen.sensor_id
        JOIN {FQN}.systems  s   ON sen.system_id = s.system_id
        JOIN {FQN}.aircraft a   ON s.aircraft_id = a.aircraft_id
        WHERE sen.type = 'EGT'
        GROUP BY a.model
        ORDER BY avg_egt DESC
    """)
)

## Step 2: Create the Genie space

Genie spaces are created in the Databricks UI, so the rest of Part A is click-through. Keep this notebook open beside the UI.

### 2.1 Navigate to AI/BI Genie

1. In your Databricks workspace, click **New** > **Genie space**
2. Or click **AI/BI** in the left sidebar, then **New Genie space**

### 2.2 Connect your data

The **Connect your data** dialog appears. Select **All Catalogs** > `databricks-neo4j-workshop` > `aircraft`, then select `sensor_readings`, `aircraft`, `sensors`, and `systems`.

> **Tip:** these tables form a join chain: `sensor_readings` -> `sensors` -> `systems` -> `aircraft`

![Connect your data dialog](https://raw.githubusercontent.com/neo4j-partners/databricks-neo4j-workshop/main/Lab_4_Compound_AI_Agents/images/genie_connect_data.png)

> **Tip:** if you do not see the tables under **Recent**, click **All** or search for `databricks-neo4j-workshop`.

### 2.3 Configure basic settings

Once the space exists, click **Configure** in the top navigation bar, then the **Settings** tab:

![Genie space Configure > Settings panel](https://raw.githubusercontent.com/neo4j-partners/databricks-neo4j-workshop/main/Lab_4_Compound_AI_Agents/images/configure_basics_genie.png)

1. **Title:** `Aircraft Sensor Analyst [YOUR_INITIALS]`
   - Example: `Aircraft Sensor Analyst RK`
2. **Description:** "Analyzes aircraft engine sensor telemetry including EGT, vibration, N1 speed, and fuel flow metrics"
3. **Default warehouse:** select a **Serverless SQL Warehouse**

### 2.4 Add sample questions

Still on the **Settings** tab, scroll to **Sample questions**. These teach Genie your domain language. Click **+ Add** and enter each of these.

**Time-Series Analytics**

```
What is the average EGT temperature for aircraft N10000 over the last 30 days?
```

**Fleet Comparisons**

```
Compare average EGT temperatures between Boeing 737 and Airbus A320 aircraft
```

**Anomaly Detection**

```
Find sensors with readings above their 95th percentile value
```

**Trend Analysis**

```
Show the trend of EGT temperatures over the 90-day period for aircraft N10000
```

## Step 3: Add Instructions

Go to **Configure** > **Instructions**. Instructions carry domain knowledge and query conventions: sensor types, normal ranges, and data conventions. Without them Genie writes plausible SQL rather than correct SQL.

Copy the whole block below into the **Instructions** field.

```
# Aircraft Sensor Analytics Domain Knowledge

## Sensor Types and Normal Ranges
- EGT (Exhaust Gas Temperature): unit column value °C. The normal range is per model, taken from each maintenance manual's takeoff limits: A320-200 620-680, A220-300 855-890, E190 870-900, B737-800 900-950, A321neo 980-1040. Always filter or group by model before comparing EGT across aircraft.
- Vibration: Normal range 0.05-0.50 inches per second, unit column value ips
- N1Speed (Fan Speed N1): Normal range 85-100, unit column value % RPM
- FuelFlow: unit column value kg/s. The normal range is per model: E190 1.00-1.20, A220-300 1.15-1.35, B737-800 1.20-1.50, A320-200 1.20-1.95, A321neo 1.50-2.00.

## Fleet Information
- Operators: ExampleAir, SkyWays, RegionalCo, NorthernJet
- Models: B737-800 by Boeing, A320-200 by Airbus, A321neo by Airbus, E190 by Embraer, A220-300 by Airbus

## Sensor Configuration
- Each aircraft has 2 engines
- Each engine has 4 sensors: EGT, Vibration, N1Speed, FuelFlow

## Data Conventions
- Timestamps are stored as timestamp type in the `timestamp` column
- Data period: July 1, 2024 to September 28, 2024 (90 days)
- Readings are every 4 hours (6 per day per sensor)

## Sensor ID Format
- Format: AC{aircraft_number}-S{system_number}-SN{sensor_number}
- Example: AC1001-S01-SN01 = Aircraft 1001, Engine 1 (S01), EGT sensor (SN01)
- S01 and S02 are always engines; S03 is Avionics; S04 is Hydraulics
- SN01=EGT, SN02=Vibration, SN03=N1Speed, SN04=FuelFlow

## Engine Names by Model
The `systems` table stores each engine as a system named after its engine model, so these are the exact strings to match on.
- B737-800: CFM56-7B
- A320-200: CFM56-5B
- A321neo: LEAP-1A
- E190: CF34-10E
- A220-300: PW1500G

## Query Conventions
- When asked about "Engine 1", filter by systems where name contains "#1"
- When asked about "Engine 2", filter by systems where name contains "#2"
- Use tail_number for human-readable aircraft references (e.g., N10000)
- Use aircraft_id for internal references (e.g., AC1001)
- Always include units in results (°C, ips, % RPM, kg/s)
```

## Step 4: Test the Genie space

### 4.1 Start a conversation

Click **Start conversation**, or open the chat interface.

### 4.2 Test basic queries

Work down this list. Each query is harder than the one before it.

**Query 1: Simple Aggregation**
```
What is the average EGT temperature across all sensors?
```
Expected: a single number around 865 degrees Celsius. The fleet mixes models whose EGT bands run from 620-680 on the A320-200 up to 980-1040 on the A321neo, so a fleet-wide average sits between them and is not meaningful on its own.

**Query 2: Filtering by Aircraft**
```
Show the average EGT for aircraft N10000
```
Expected: average EGT for that specific aircraft

**Query 3: Time-Series Trend**
```
Show daily average EGT for aircraft AC1001 in July 2024
```
Expected: about 30 rows with a date and an average value

**Query 4: Cross-Table Join**
```
Compare average vibration readings by aircraft model
```
Expected: results grouped by B737-800, A320-200, A321neo, E190

**Query 5: Statistical Analysis**
```
Find the top 5 sensors with the highest average readings for their type
```
Expected: top sensors with their average values and types

### 4.3 Read the generated SQL

For each answer, click **View Code** and check the query Genie wrote.

Here is what a correct answer to "Compare average vibration by aircraft model" looks like:

```sql
SELECT
    a.model,
    AVG(r.value) as avg_vibration,
    COUNT(*) as reading_count
FROM sensor_readings r
JOIN sensors sen ON r.sensor_id = sen.sensor_id
JOIN systems s ON sen.system_id = s.system_id
JOIN aircraft a ON s.aircraft_id = a.aircraft_id
WHERE sen.type = 'Vibration'
GROUP BY a.model
ORDER BY avg_vibration DESC
```

Run the same query here and compare the numbers to what Genie returned.

In [ ]:
display(
    spark.sql(f"""
        SELECT
            a.model,
            AVG(r.value) AS avg_vibration,
            COUNT(*)     AS reading_count
        FROM {FQN}.sensor_readings r
        JOIN {FQN}.sensors  sen ON r.sensor_id = sen.sensor_id
        JOIN {FQN}.systems  s   ON sen.system_id = s.system_id
        JOIN {FQN}.aircraft a   ON s.aircraft_id = a.aircraft_id
        WHERE sen.type = 'Vibration'
        GROUP BY a.model
        ORDER BY avg_vibration DESC
    """)
)

## Step 5: Save and record the Genie space ID

### 5.1 Save

Click **Save** to keep your configuration.

### 5.2 Record the name

Note the exact name of your space, for example `Aircraft Sensor Analyst RK`. Part B refers to it when the instructor adds it as a subagent.

### 5.3 Record the space ID

**Lab 5 needs this ID.** It is in the browser URL of your Genie space:

```
https://<your-workspace>.cloud.databricks.com/genie/rooms/01f0a1b2c3d4e5f6a7b8c9d0e1f2a3b4
                                                          ^-------------------------------^
                                                          this is the Genie space ID
```

Paste it into the cell below and run it. Keep this notebook, or copy the value somewhere you will find it at the start of Lab 5.

In [ ]:
# Your Genie space ID, from the URL: .../genie/rooms/<GENIE_SPACE_ID>
# Lab 5 uses this to attach the Genie space as one of the agent's three tools.
GENIE_SPACE_ID = ""

if not GENIE_SPACE_ID:
    print("Not set yet. Open your Genie space and copy the id out of the URL.")
else:
    print(f"Genie space ID recorded: {GENIE_SPACE_ID}")

## Part A Summary

Your Genie space now:

- Answers questions about sensor telemetry in plain English
- Aggregates by aircraft, model, operator, or sensor type
- Runs statistical analysis: averages, percentiles, standard deviation
- Joins across the four tables to give context-rich answers
- Understands the domain vocabulary: EGT, N1Speed, and the rest

### More sample queries

Add these as sample questions, or use them to test further.

**Time-Series Analytics**

```
Show daily average vibration readings for Engine 1 on aircraft AC1001
```

```
What was the maximum fuel flow recorded in August 2024?
```

**Fleet Comparisons**

```
Which aircraft has the highest average vibration readings?
```

```
Show fuel flow rates by operator
```

**Anomaly Detection**

```
Show all EGT readings above 950 degrees Celsius for B737-800 aircraft
```

```
Which engines have N1 speed readings outside the normal range of 85-100% RPM?
```

**Trend Analysis**

```
Calculate the 7-day rolling average of vibration for Engine 1 on AC1001
```

### Where Genie stops

Genie answers *how much* and *how often*. Average EGT on a tail number over 30 days, the maximum fuel flow in August, which aircraft vibrates most. Every one of those is an aggregation over timestamped rows, and SQL over the Lakehouse is the right tool for all of them.

Ask it "which component failure delayed which flight" and it has nothing to work with. That question is a traversal: component to maintenance event to flight to delay, following relationships rather than scanning a column. No amount of instruction tuning produces it, because the relationships live in Neo4j and Genie queries Unity Catalog.

There are two ways to give an agent both. You build one in Lab 5. You watch the other now.

---

# Part B: Multi-Agent Supervisor (Instructor Demo)

> ## Watch this one. Do not build it.
>
> **There is nothing to run in this section.** Every block below is reference text, not code cells. Follow along on the instructor's screen.
>
> The demo runs against **the instructor's demo workspace** and **the instructor's reference Aura instance**. You need nothing from your own account:
> - **No Aura instance.** The demo queries the instructor's graph, not yours.
> - **No Unity Catalog MCP connection.** You do not create one. It lives in the instructor's workspace.
> - **No OAuth credential.** The client id and secret belong to the instructor's setup and are never handed out.
>
> **Why it is in the workshop.** The demo shows the same routing architecture as Lab 5, built with no code and with centrally governed access to Neo4j. Agent Bricks Multi-Agent Supervisor reaches a working system through configuration alone. A Unity Catalog HTTP connection with OAuth2 M2M against a hosted MCP server is what governed agent access to Neo4j looks like in production. Seeing one architecture from two directions is the point.

The supervisor routes each question to either the **Genie space** for sensor analytics or the **Neo4j MCP agent** for graph relationships. Questions that need both get routed to both.

**Demo runtime:** about 20 minutes on screen.

**The graph behind the demo.** The MCP server points at the instructor's demo Aura instance, loaded before class with `workshop-setup/populate_aircraft_db`. It holds the complete dataset: the full fleet with all systems, components, sensors, maintenance events, flights, and delays.

## What the instructor builds, part 1: the Neo4j MCP connection

A Unity Catalog **external connection** named `neo4j_agentcore_mcp` points at a hosted Neo4j MCP server. Governance sits in Unity Catalog: access to Neo4j is a `USE CONNECTION` grant, not a password in a notebook.

The connection is checked under **Catalog** > **External connections** before class.

The MCP server exposes two tools:

| Tool | What it does |
|------|--------------|
| `get_neo4j_schema` | Returns the graph schema: labels, relationship types, and properties |
| `read_neo4j_cypher` | Runs read-only Cypher queries |

The AgentCore Gateway prefixes both names, so in the tool list they appear as `neo4j-mcp-server-target___get_neo4j_schema` and `neo4j-mcp-server-target___read_neo4j_cypher`.

The instructor smoke-tests them in **AI Playground** with **GPT OSS 120B**, adding the tool through **Add your own tool** > **+ Add tool** > **MCP Servers** > **External MCP Servers** > **neo4j_agentcore_mcp**, then asking:

```
What is the schema of the database?
Which aircraft had critical maintenance events?
```

## What the instructor builds, part 2: the Supervisor and its two subagents

Under **Agents** > **Multi-Agent Supervisor** > **Build**:

- **Name:** `Aircraft Intelligence Hub [YOUR_INITIALS]`
- **Description:** "Intelligent coordinator for aircraft analytics combining sensor telemetry data from Unity Catalog with knowledge graph relationships from Neo4j."

### Subagent 1: the Neo4j graph agent

Added under **Configure Agents** with **Type** set to **External MCP Server** and the **Unity Catalog connection** set to `neo4j_agentcore_mcp`. The **Agent Name** `mcp-neo4j-agentcore-mcp` auto-populates.

![MCP Connection Configuration](https://raw.githubusercontent.com/neo4j-partners/databricks-neo4j-workshop/main/Lab_4_Compound_AI_Agents/images/mcp_connection.png)

The supervisor reads the **Describe the content** field to decide which agent handles each question, so the detail in it is load-bearing:

```
Queries the Neo4j knowledge graph to explore aircraft relationships, topology, and operational data.

BEST FOR:
- Aircraft topology: "What systems does aircraft AC1001 have?"
- Component hierarchy: "Show all components in the hydraulics system"
- Maintenance events: "Which aircraft had critical maintenance events?"
- Flight operations: "Find flights delayed due to maintenance"
- Relationship patterns: "Which airports does ExampleAir fly to?"
- Graph traversals: "Show the path from aircraft to sensor"

DATA AVAILABLE (loaded from /Volumes/databricks-neo4j-workshop/aircraft/raw_data/):
- Aircraft: Fleet inventory with tail numbers, models, operators
- Systems: Engines, Avionics, Hydraulics per aircraft
- Components: Turbines, Compressors, Pumps, etc.
- Sensors: Monitoring equipment metadata
- MaintenanceEvents: Faults, severity, corrective actions
- Flights: Operations with departure/arrival
- Delays: Delay causes and durations
- Airports: Route network locations

RELATIONSHIP TYPES:
- HAS_SYSTEM: Aircraft -> System
- HAS_COMPONENT: System -> Component
- HAS_SENSOR: System -> Sensor
- HAS_EVENT: Component -> MaintenanceEvent
- OPERATES_FLIGHT: Aircraft -> Flight
- DEPARTS_FROM / ARRIVES_AT: Flight -> Airport
- HAS_DELAY: Flight -> Delay

DO NOT USE FOR:
- Time-series sensor readings (use sensor_data_agent instead)
- Statistical aggregations over readings
- Trend analysis or rolling averages
```

### Subagent 2: the Genie space agent

Added with **Type** set to **Genie space**, pointing at the space built in Part A, `Aircraft Sensor Analyst [YOUR_INITIALS]`, and named `sensor_data_agent`. Its description:

```
Analyzes aircraft sensor telemetry data using SQL queries over Unity Catalog tables.

DATA LOCATION:
- Catalog: databricks-neo4j-workshop
- Schema: aircraft
- Tables: sensor_readings, sensors, systems, aircraft

BEST FOR:
- Time-series analytics: "What is the average EGT over the last 30 days?"
- Statistical analysis: "Show sensors above the 95th percentile"
- Trend detection: "Show daily vibration trends for Engine 1"
- Fleet comparisons: "Compare fuel flow between Boeing and Airbus"
- Anomaly detection: "Find B737-800 EGT readings above 950 degrees"
- Aggregations: "What was the maximum N1 speed recorded?"

DATA AVAILABLE:
- sensor_readings: Telemetry every 4 hours over 90 days
- sensors: Sensor metadata (type, unit, system)
- systems: Aircraft system information
- aircraft: Fleet metadata (model, operator)

SENSOR TYPES:
- EGT: Exhaust Gas Temperature (unit column value °C). Per model: A320-200 620-680, A220-300 855-890, E190 870-900, B737-800 900-950, A321neo 980-1040
- Vibration: Engine vibration (0.05-0.50, unit column value ips)
- N1Speed: Fan speed (85-100, unit column value % RPM)
- FuelFlow: Fuel consumption (unit column value kg/s). Per model: E190 1.00-1.20, A220-300 1.15-1.35, B737-800 1.20-1.50, A320-200 1.20-1.95, A321neo 1.50-2.00

DO NOT USE FOR:
- Relationship queries (use mcp-neo4j-agentcore-mcp)
- Maintenance event details
- Flight operations or delays
- Component-level fault tracking
```

The instructor clicks **Create Agent**. Deployment takes several minutes.

## What the instructor builds, part 3: the routing instructions

After the agent deploys, the instructor expands the **Optional** section at the bottom of the configuration page and pastes this into **Instructions**, then clicks **Update Agent**.

Read it. This is the whole routing policy, and Lab 5 rebuilds the same thing as a Python supervisor node.

```
# Aircraft Intelligence Hub - Routing Instructions

You are an intelligent coordinator for aircraft analytics. Your role is to understand user questions and route them to the appropriate specialized agent.

## Available Agents

### sensor_data_agent (Genie space - Unity Catalog SQL)
Use for questions about:
- Sensor readings and telemetry data
- Time-series analytics (averages, trends, rolling windows)
- Statistical analysis (percentiles, standard deviation)
- Fleet-wide comparisons of sensor metrics
- Anomaly detection based on readings
- Questions containing: EGT, vibration, N1, fuel flow, temperature, readings, averages, trends

### mcp-neo4j-agentcore-mcp (Neo4j Knowledge Graph - Cypher)
Use for questions about:
- Aircraft structure and topology
- Component relationships and hierarchy
- Maintenance events and fault history
- Flight operations, routes, delays
- "Which", "what systems", "connected to", "related to" questions
- Questions about maintenance, flights, delays, airports

## Routing Rules

1. **Sensor values/readings** -> sensor_data_agent
   - "What is the EGT for..."
   - "Show vibration readings..."
   - "Average fuel flow..."

2. **Relationships/structure** -> mcp-neo4j-agentcore-mcp
   - "What systems does aircraft X have?"
   - "Which components..."
   - "Show maintenance events..."

3. **Flights/operations** -> mcp-neo4j-agentcore-mcp
   - "Which flights were delayed?"
   - "What airports does..."
   - "Show flight routes..."

4. **Maintenance history** -> mcp-neo4j-agentcore-mcp
   - "What maintenance events..."
   - "Which components had faults?"
   - "Critical maintenance..."

5. **Statistical aggregations on readings** -> sensor_data_agent
   - "Average", "maximum", "minimum", "percentile"
   - "Trend", "over time", "daily", "monthly"
   - "Compare", "between", "by model"

## Complex Queries (Multi-Agent)

For questions that need BOTH sources, process sequentially:

1. **"Find aircraft with high vibration AND recent maintenance"**
   - First: sensor_data_agent -> Get aircraft with high vibration
   - Then: mcp-neo4j-agentcore-mcp -> Get maintenance events for those aircraft

2. **"Which engines have abnormal EGT and what components were serviced?"**
   - First: sensor_data_agent -> Find abnormal EGT readings
   - Then: mcp-neo4j-agentcore-mcp -> Find maintenance events for those engines

3. **"Compare sensor trends for aircraft that had delays"**
   - First: mcp-neo4j-agentcore-mcp -> Get aircraft with delays
   - Then: sensor_data_agent -> Get sensor trends for those aircraft

## Response Guidelines

1. For single-agent queries: Return the agent's response directly
2. For multi-agent queries: Synthesize a combined response that integrates both perspectives
3. Always cite which data source provided each piece of information
4. If a query cannot be answered by either agent, explain what data would be needed
```

## What to watch for: single-agent routing

The instructor tests in the **Test your Agent** panel on the Build tab, or in **Open in Playground**. Watch which subagent the supervisor picks before you look at the answer.

**Test 1: sensor analytics, should route to `sensor_data_agent`**
```
What is the average EGT temperature across the fleet?
```
Good answer: a numerical average, produced by SQL over `sensor_readings`.

**Test 2: graph relationships, should route to `mcp-neo4j-agentcore-mcp`**
```
What systems does aircraft AC1001 have?
```
Good answer: the engine, avionics, and hydraulics systems, produced by Cypher.

**Test 3: maintenance events, should route to `mcp-neo4j-agentcore-mcp`**
```
Show me all critical maintenance events in the last month
```
Good answer: maintenance events with `severity=CRITICAL`.

**Test 4: fleet comparison, should route to `sensor_data_agent`**
```
Compare average vibration readings between Boeing and Airbus aircraft
```
Good answer: statistics grouped by manufacturer.

## What to watch for: multi-agent queries

These are the point of the whole demo. Neither subagent can answer them alone.

**Test 5: combined query**
```
Find B737-800 aircraft with EGT readings above 950 degrees and show their maintenance history
```
Good answer, in three visible moves:
1. The supervisor asks `sensor_data_agent` for the high-EGT aircraft
2. It asks `mcp-neo4j-agentcore-mcp` for the maintenance history of exactly those aircraft
3. It writes one answer that cites both sources

**Test 6: combined query, the other direction**
```
Which engines had above-average vibration, and what components were recently serviced on those engines?
```
Good answer:
1. High-vibration engines from `sensor_data_agent`
2. Maintenance events for those engines from `mcp-neo4j-agentcore-mcp`
3. Results combined into one response

A weak answer answers only half the question, or answers the second half about the whole fleet rather than about the aircraft the first half returned. Watch for that.

### More queries the instructor may show

**Sensor analytics, `sensor_data_agent`**
```
What is the average EGT temperature for aircraft N10000?
Show daily vibration trends for Engine 1 in August 2024
Find all sensors with readings above the 95th percentile
Compare fuel flow rates by aircraft model
What was the maximum N1 speed recorded?
```

**Graph queries, `mcp-neo4j-agentcore-mcp`**
```
What systems does aircraft AC1001 have?
Show all components in the hydraulics system
Which aircraft had critical maintenance events?
Find flights that were delayed due to maintenance
What airports are in the route network?
```

**Combined, both agents**
```
Find aircraft with high EGT and show their recent maintenance
Which engines have abnormal vibration and what was serviced?
Compare sensor trends for aircraft that had delays vs. those that didn't
Show maintenance events for aircraft with the lowest fuel efficiency
```

## The supervisor is also an endpoint

The deployed supervisor is a serving endpoint, so anything that can post JSON can use it. Under **Open in Playground** > **Get code**, Databricks hands back a curl or Python snippet.

This is reference only. Do not run it, the endpoint is in the instructor's workspace.

```python
import requests

# Get your endpoint URL from the agent status page
ENDPOINT_URL = "https://<workspace>.databricks.com/serving-endpoints/<agent-name>/invocations"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

response = requests.post(
    ENDPOINT_URL,
    headers=headers,
    json={"messages": [{"role": "user", "content": "What systems does AC1001 have?"}]}
)
print(response.json())
```

Lab 5 deploys your own agent to an endpoint that looks just like this one, built in code instead of configuration.

## Part B Summary

The demo combines two purpose-built data platforms behind one question box:

- **Genie plus Lakehouse for time-series data.** SQL analytics over sensor telemetry, right for aggregations, trends, and statistics.
- **Neo4j for relationship data.** Graph traversals across aircraft topology, maintenance events, flights, and delays, right for multi-hop questions.
- **Intelligent routing.** The supervisor sends each question to the right source on its own.
- **Cross-source synthesis.** Questions that span both get answered by querying each in turn and combining the results.
- **Natural language access.** Users need neither SQL nor Cypher.

The instructor can also refine the supervisor after the fact: the **Examples** tab takes labeled test questions, and **Guidelines** on those examples correct routing mistakes without touching the instructions block.

---

## Next Steps

Continue to **Lab 5**. It builds a LangGraph supervisor in Python over three tools:

1. The Genie space you created in Part A, addressed by the `GENIE_SPACE_ID` you recorded above
2. Cypher over **your own** Aura instance, the one you loaded in Lab 2
3. The GraphRAG retrievers you built in Lab 3

It ends with the agent deployed to Model Serving and authenticating as a service principal.

Lab 5 and Part B are the same architecture seen from two directions. Part B reaches it through configuration with governance in Unity Catalog. Lab 5 reaches it in code, against the graph you loaded yourself.

**Before you leave this notebook:** check that `GENIE_SPACE_ID` above is filled in. Lab 5 starts by asking for it.

After the workshop:
- Add a third subagent for maintenance procedure search
- Build custom Unity Catalog functions as extra tools
- Add guardrails and output validation
- Give the agent memory in Neo4j, which is **Lab 6**